# Week 4 – Bronze to Silver

This notebook reads the four Bronze Delta tables created in `Week4_Source_to_Bronze.ipynb`, applies basic Silver-layer cleansing and standardization, removes duplicate records, and writes four Silver Delta tables.

**Bronze tables used**
- `workspace.default.bronze_users`
- `workspace.default.bronze_subscriptions`
- `workspace.default.bronze_sessions`
- `workspace.default.bronze_content_catalog`

**Silver tables created**
- `workspace.default.silver_users`
- `workspace.default.silver_subscriptions`
- `workspace.default.silver_sessions`
- `workspace.default.silver_content_catalog`


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# Common helper functions for Silver-layer cleansing
def clean_strings(df):
    # Trim whitespace from all string columns and convert empty strings to NULL
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(
                field.name,
                F.when(F.trim(F.col(field.name)) == "", None)
                 .otherwise(F.trim(F.col(field.name)))
            )
    return df

def add_silver_timestamp(df):
    return df.withColumn("silver_processed_time", F.current_timestamp())

def write_silver(df, table_name):
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(table_name))


## 1. Silver Users

In [ ]:
bronze_users = spark.table("workspace.default.bronze_users")

print("Bronze users:", bronze_users.count())
bronze_users.printSchema()

silver_users = clean_strings(bronze_users)

# Remove duplicate user records.
# If user_id exists, use it as the business key; otherwise remove exact duplicates.
if "user_id" in silver_users.columns:
    silver_users = silver_users.dropDuplicates(["user_id"])
else:
    silver_users = silver_users.dropDuplicates()

silver_users = add_silver_timestamp(silver_users)

# Keep ingestion_time as Bronze metadata and add Silver processing time.
write_silver(silver_users, "workspace.default.silver_users")

display(silver_users)


## 2. Silver Subscriptions

In [ ]:
bronze_subscriptions = spark.table("workspace.default.bronze_subscriptions")

print("Bronze subscriptions:", bronze_subscriptions.count())
bronze_subscriptions.printSchema()

silver_subscriptions = clean_strings(bronze_subscriptions)

# Standardize subscription date columns when they are present.
for col_name in ["period_start_date", "period_end_date"]:
    if col_name in silver_subscriptions.columns:
        silver_subscriptions = silver_subscriptions.withColumn(
            col_name, F.to_date(F.col(col_name))
        )

# Normalize common flag/status fields.
if "auto_renew_flag" in silver_subscriptions.columns:
    silver_subscriptions = silver_subscriptions.withColumn(
        "auto_renew_flag",
        F.col("auto_renew_flag").cast("boolean")
    )

if "lifecycle_status" in silver_subscriptions.columns:
    silver_subscriptions = silver_subscriptions.withColumn(
        "lifecycle_status", F.upper(F.col("lifecycle_status"))
    )

# Remove duplicate subscriptions using subscription_id.
if "subscription_id" in silver_subscriptions.columns:
    silver_subscriptions = silver_subscriptions.dropDuplicates(["subscription_id"])
else:
    silver_subscriptions = silver_subscriptions.dropDuplicates()

silver_subscriptions = add_silver_timestamp(silver_subscriptions)

write_silver(
    silver_subscriptions,
    "workspace.default.silver_subscriptions"
)

display(silver_subscriptions)


## 3. Silver Sessions

In [ ]:
bronze_sessions = spark.table("workspace.default.bronze_sessions")

print("Bronze sessions:", bronze_sessions.count())
bronze_sessions.printSchema()

silver_sessions = clean_strings(bronze_sessions)

# Convert likely timestamp columns to timestamp type when they are stored as strings.
timestamp_candidates = [
    "session_start_time",
    "session_end_time",
    "start_time",
    "end_time",
    "event_time",
    "timestamp"
]

for col_name in timestamp_candidates:
    if col_name in silver_sessions.columns:
        silver_sessions = silver_sessions.withColumn(
            col_name, F.to_timestamp(F.col(col_name))
        )

# Remove exact duplicate session records.
# Prefer session_id when it exists.
if "session_id" in silver_sessions.columns:
    silver_sessions = silver_sessions.dropDuplicates(["session_id"])
else:
    silver_sessions = silver_sessions.dropDuplicates()

silver_sessions = add_silver_timestamp(silver_sessions)

write_silver(
    silver_sessions,
    "workspace.default.silver_sessions"
)

display(silver_sessions)


## 4. Silver Content Catalog

In [ ]:
bronze_catalog = spark.table("workspace.default.bronze_content_catalog")

print("Bronze content catalog:", bronze_catalog.count())
bronze_catalog.printSchema()

silver_catalog = clean_strings(bronze_catalog)

# Remove duplicate catalog records.
# Prefer content_id when available; otherwise use exact duplicate removal.
if "content_id" in silver_catalog.columns:
    silver_catalog = silver_catalog.dropDuplicates(["content_id"])
elif "catalog_id" in silver_catalog.columns:
    silver_catalog = silver_catalog.dropDuplicates(["catalog_id"])
else:
    silver_catalog = silver_catalog.dropDuplicates()

silver_catalog = add_silver_timestamp(silver_catalog)

write_silver(
    silver_catalog,
    "workspace.default.silver_content_catalog"
)

display(silver_catalog)


## 5. Verify Silver Tables

In [ ]:
spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)

tables = [
    "workspace.default.silver_users",
    "workspace.default.silver_subscriptions",
    "workspace.default.silver_sessions",
    "workspace.default.silver_content_catalog"
]

for table_name in tables:
    print("\n" + "=" * 70)
    print(table_name)
    print("=" * 70)
    df = spark.table(table_name)
    print("Records:", df.count())
    df.printSchema()
    display(df.limit(10))


## Silver-layer processing performed

1. Read data from the existing Bronze Delta tables.
2. Trim leading/trailing whitespace from string columns.
3. Convert empty strings to `NULL`.
4. Standardize subscription date and status/flag fields.
5. Convert common session time fields to timestamps when those columns exist.
6. Remove duplicate records using business IDs when available.
7. Add `silver_processed_time` for Silver-layer processing tracking.
8. Save the cleaned datasets as Delta tables in the `workspace.default` schema.

The code is intentionally defensive: it checks whether optional columns such as `user_id`, `session_id`, `content_id`, or timestamp fields exist before applying transformations.
